In [0]:
dbutils.fs.ls("/Volumes/workspace/default/healthcare_cyber_raw/")

[FileInfo(path='dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv', name='hhs_breach_report.csv', size=93280, modificationTime=1778684439000)]

In [0]:
raw_path = "/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv"

hhs_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .csv(raw_path)
)

display(hhs_df)

javax.faces.component.UIPanel@6972d666,State,Covered Entity Type,Individuals Affected,Breach Submission Date,Type of Breach,Location of Breached Information,javax.faces.component.UIPanel@48c348d2,Web Description
University of Michigan/Michigan Medicine,MI,Healthcare Provider,551,2026-05-01,Unauthorized Access/Disclosure,Electronic Medical Record,No,null
Mt. Spokane Pediatrics,WA,Healthcare Provider,32021,2026-04-30,Hacking/IT Incident,Network Server,No,null
"Ouster, Inc.",CA,Health Plan,574,2026-04-30,Unauthorized Access/Disclosure,Network Server,No,null
Northwoods Surgery Center,MN,Healthcare Provider,5385,2026-04-29,Hacking/IT Incident,Network Server,No,null
Tri-Cities Gastroenterology,TN,Healthcare Provider,67115,2026-04-29,Hacking/IT Incident,Network Server,No,null
Belmont Aesthetic and Reconstructive Plastic Surgery,VA,Healthcare Provider,528,2026-04-23,Hacking/IT Incident,Network Server,No,null
Interim HealthCare of Lubbock,TX,Healthcare Provider,2071,2026-04-23,Hacking/IT Incident,Network Server,Yes,null
Interim HealthCare of Amarillo,TX,Healthcare Provider,666,2026-04-23,Hacking/IT Incident,Network Server,Yes,null
Liberty Bankers Life Ins. Co.,TX,Health Plan,20202,2026-04-22,Hacking/IT Incident,Network Server,Yes,null
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null


In [0]:
print(hhs_df.columns)

['javax.faces.component.UIPanel@6972d666', 'State', 'Covered Entity Type', 'Individuals Affected', 'Breach Submission Date', 'Type of Breach', 'Location of Breached Information', 'javax.faces.component.UIPanel@48c348d2', 'Web Description']


In [0]:
from pyspark.sql.functions import current_timestamp, col

# First column has a wrong exported name, so we rename it
first_col = hhs_df.columns[0]

bronze_df = (
    hhs_df
    .withColumnRenamed(first_col, "covered_entity_name")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("Covered Entity Type", "covered_entity_type")
    .withColumnRenamed("Individuals Affected", "individuals_affected")
    .withColumnRenamed("Breach Submission Date", "breach_submission_date")
    .withColumnRenamed("Type of Breach", "type_of_breach")
    .withColumnRenamed("Location of Breached Information", "location_of_breached_information")
    .withColumnRenamed("Business Associate Present", "business_associate_present")
    .withColumnRenamed("Web Description", "web_description")
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file_path", col("_metadata.file_path"))
)

display(bronze_df)


covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,javax.faces.component.UIPanel@48c348d2,web_description,ingestion_time,source_file_path
University of Michigan/Michigan Medicine,MI,Healthcare Provider,551,2026-05-01,Unauthorized Access/Disclosure,Electronic Medical Record,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Mt. Spokane Pediatrics,WA,Healthcare Provider,32021,2026-04-30,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
"Ouster, Inc.",CA,Health Plan,574,2026-04-30,Unauthorized Access/Disclosure,Network Server,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Northwoods Surgery Center,MN,Healthcare Provider,5385,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Tri-Cities Gastroenterology,TN,Healthcare Provider,67115,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Belmont Aesthetic and Reconstructive Plastic Surgery,VA,Healthcare Provider,528,2026-04-23,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Interim HealthCare of Lubbock,TX,Healthcare Provider,2071,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Interim HealthCare of Amarillo,TX,Healthcare Provider,666,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Liberty Bankers Life Ins. Co.,TX,Health Plan,20202,2026-04-22,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null,2026-05-16T14:46:04.931Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv


In [0]:
#saving Bronze dataframe as Delta table
bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_hhs_healthcare_breaches"
)

In [0]:
display(spark.sql("""
SELECT *
FROM workspace.default.bronze_hhs_healthcare_breaches

"""))

covered_entity_name,state,covered_entity_type,individuals_affected,breach_submission_date,type_of_breach,location_of_breached_information,javax.faces.component.UIPanel@48c348d2,web_description,ingestion_time,source_file_path
University of Michigan/Michigan Medicine,MI,Healthcare Provider,551,2026-05-01,Unauthorized Access/Disclosure,Electronic Medical Record,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Mt. Spokane Pediatrics,WA,Healthcare Provider,32021,2026-04-30,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
"Ouster, Inc.",CA,Health Plan,574,2026-04-30,Unauthorized Access/Disclosure,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Northwoods Surgery Center,MN,Healthcare Provider,5385,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Tri-Cities Gastroenterology,TN,Healthcare Provider,67115,2026-04-29,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Belmont Aesthetic and Reconstructive Plastic Surgery,VA,Healthcare Provider,528,2026-04-23,Hacking/IT Incident,Network Server,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Interim HealthCare of Lubbock,TX,Healthcare Provider,2071,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Interim HealthCare of Amarillo,TX,Healthcare Provider,666,2026-04-23,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
Liberty Bankers Life Ins. Co.,TX,Health Plan,20202,2026-04-22,Hacking/IT Incident,Network Server,Yes,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv
"Spokane Digestive Disease Center, P.S.",WA,Healthcare Provider,501,2026-04-20,Hacking/IT Incident,Email,No,null,2026-05-16T14:51:49.733Z,dbfs:/Volumes/workspace/default/healthcare_cyber_raw/hhs_breach_report.csv


In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM workspace.default.bronze_hhs_healthcare_breaches
""").show()

+-------------+
|total_records|
+-------------+
|          725|
+-------------+



In [0]:
#Dataset downloaded: Done
#Uploaded to Databricks Volume: Done
#Read CSV into Spark DataFrame: Done
#Renamed raw columns: Done
#Added ingestion metadata: Done
#Next: Save Bronze Delta table